# Iris Walkthrough — Setup to Scripts

Objectif: montrer la progression locale/notebook avant le passage en scripts et CI/CD.

Etapes:
1. Setup imports + chemins
2. Prep des donnees Iris
3. Train du modele
4. Evaluation + quality gate
5. Transition vers execution script et pipeline

In [ ]:
import os
import sys
import tempfile
import pandas as pd

sys.path.insert(0, '../src')
from prep import main as prep_main
from train import main as train_main
from evaluate import main as evaluate_main

## 1) Setup des dossiers de travail

In [ ]:
workdir = tempfile.mkdtemp(prefix='iris-nb-')
data_dir = os.path.join(workdir, 'data')
model_dir = os.path.join(workdir, 'model')
workdir, data_dir, model_dir

## 2) Prep (equivalent `prep.py`)

In [ ]:
class PrepArgs:
    def __init__(self, output_dir):
        self.output_dir = output_dir

prep_main(PrepArgs(data_dir))

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
print(train_df.shape, test_df.shape)
train_df.head()

## 3) Train (equivalent `train.py`)

In [ ]:
class TrainArgs:
    def __init__(self, data_dir, model_dir, max_iter=200):
        self.data_dir = data_dir
        self.model_dir = model_dir
        self.max_iter = max_iter

train_main(TrainArgs(data_dir, model_dir, max_iter=200))
os.listdir(model_dir)

## 4) Evaluate (equivalent `evaluate.py`)

In [ ]:
class EvalArgs:
    def __init__(self, data_dir, model_dir, min_accuracy=0.90):
        self.data_dir = data_dir
        self.model_dir = model_dir
        self.min_accuracy = min_accuracy

evaluate_main(EvalArgs(data_dir, model_dir, min_accuracy=0.90))

## 5) Transition vers scripts et CI/CD

Apres la validation en notebook, passer aux commandes scripts:

```bash
python mlops/data-science/src/prep.py --output_dir /tmp/iris
python mlops/data-science/src/train.py --data_dir /tmp/iris --model_dir /tmp/model
python mlops/data-science/src/evaluate.py --data_dir /tmp/iris --model_dir /tmp/model
pytest tests/ -v
```

Ensuite, le workflow `ci-train.yml` execute le meme enchainement automatiquement dans GitHub Actions.